# Tanshi — Production Voice Cloning · Cloud Setup

**Run All.** This notebook prepares a reproducible Colab environment for voice-model
work: mounts Drive, clones the repo, installs dependencies, auto-finds the production
voice dataset, and validates everything against a committed checksum manifest.

**The only thing you ever edit is the `MODEL` line in the Config cell below.**
Future benchmark phases just select a different model name and re-run — nothing else changes.

> This notebook performs **setup + validation only**. It does not train or benchmark.


In [ ]:
# ============================ CONFIG — the only cell you edit ================
# Future benchmark phases: change MODEL only.
MODEL = "xtts-v2"

AVAILABLE_MODELS = [
    "xtts-v2", "f5-tts", "styletts2", "chatterbox",
    "openvoice-v2", "melotts", "kokoro", "indic-parler", "dia",
]

REPO_URL = "https://github.com/nahatadhananjay33-svg/ai-creator-platform.git"
BRANCH   = "feat/cloud-setup"          # branch holding cloud_setup + manifest
REPO_DIR = "/content/ai-creator-platform"

# Dataset is auto-discovered under MyDrive by folder name; override if you moved it.
DATASET_DIR_NAME = "production_voice_dataset"
DATASET_PATH_OVERRIDE = ""             # e.g. "/content/drive/MyDrive/foo/production_voice_dataset"

VERIFY_MODE = "quick"                  # "quick" (presence+size, fast) or "full" (sha256)

assert MODEL in AVAILABLE_MODELS, f"MODEL must be one of {AVAILABLE_MODELS}"
print(f"MODEL = {MODEL}")
print(f"repo  = {REPO_URL} @ {BRANCH}")


In [ ]:
# ============================ 1. Mount Google Drive =========================
from google.colab import drive
drive.mount("/content/drive")

def remount():
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print("remount failed:", e)

import os
assert os.path.exists("/content/drive/MyDrive"), "Drive did not mount"
print("OK  Drive mounted")


In [ ]:
# ============================ 2. Clone repo + dependencies ==================
import subprocess, sys, shutil, os

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "clone", "-q", REPO_URL, REPO_DIR])
sh(["git", "-C", REPO_DIR, "fetch", "-q", "origin", BRANCH])
sh(["git", "-C", REPO_DIR, "checkout", "-q", BRANCH])
sh(["git", "-C", REPO_DIR, "pull", "-q", "origin", BRANCH])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "openpyxl", "tqdm", "soundfile"], check=False)
if not shutil.which("ffmpeg"):
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

commit = sh(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).stdout.strip()
print(f"OK  repo {REPO_DIR} @ {BRANCH} ({commit})")
print("OK  ffmpeg:", shutil.which("ffmpeg"))


In [ ]:
# ============================ 3. Locate the dataset =========================
from pathlib import Path

def find_dataset():
    if DATASET_PATH_OVERRIDE:
        p = Path(DATASET_PATH_OVERRIDE)
        return p if p.is_dir() else None
    root = Path("/content/drive/MyDrive")
    # shallow-first search so it stays fast on a big Drive
    for depth in range(1, 5):
        for p in root.glob("/".join(["*"] * depth)):
            try:
                if p.is_dir() and p.name == DATASET_DIR_NAME:
                    return p
            except OSError:
                continue
    return None

DATASET = find_dataset()
assert DATASET is not None, (
    f"Could not find '{DATASET_DIR_NAME}' under MyDrive. Upload it, or set "
    f"DATASET_PATH_OVERRIDE in the Config cell.")
print("OK  dataset:", DATASET)
for sub in ("accepted_segments", "rejected_segments", "metadata"):
    n = len(list((DATASET / sub).glob("*"))) if (DATASET / sub).is_dir() else 0
    print(f"      {sub:18} {n} files")


In [ ]:
# ============================ 4. Environment validation =====================
import platform, subprocess, sys, shutil, json
from pathlib import Path

checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok), detail))
    print(f"{'OK  ' if ok else 'FAIL'} {name:24} {detail}")

check("python", sys.version_info >= (3, 9), platform.python_version())
try:
    import torch
    check("torch", True, f"{torch.__version__} cuda={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        check("gpu", True, torch.cuda.get_device_name(0))
    else:
        check("gpu", False, "no CUDA runtime (set Runtime -> GPU for benchmarks)")
except Exception as e:
    check("torch", False, f"not installed ({e})")
check("ffmpeg", shutil.which("ffmpeg") is not None, shutil.which("ffmpeg") or "missing")
check("repo import", Path(REPO_DIR, "production", "cloud_setup").is_dir(), REPO_DIR)

# dataset integrity against the committed manifest
man_path = Path(REPO_DIR) / "production/cloud_setup/manifest/voice_dataset_manifest.json"
check("manifest present", man_path.exists(), str(man_path))

sys.path.insert(0, REPO_DIR)
from production.cloud_setup.manifest import load_manifest, verify_against

man = load_manifest(man_path)
res = verify_against(man, DATASET, quick=(VERIFY_MODE == "quick"), progress=True)
check("dataset integrity", res["ok"],
      f"{res['verified']}/{res['expected_files']} files "
      f"(missing {res['missing_count']}, size-bad {res['size_mismatch_count']}, "
      f"hash-bad {res['hash_mismatch_count']})")

meta = man["metadata"]
print(f"\n     accepted segments : {meta['accepted_rows']}")
print(f"     accepted hours    : {meta['accepted_hours']} (speech {meta['accepted_speech_hours']})")


In [ ]:
# ============================ 5. Ready ======================================
ok = all(c[1] for c in checks if c[0] != "gpu")   # GPU optional for setup
print("=" * 58)
print("  ENVIRONMENT READY" if ok else "  SETUP INCOMPLETE — see FAIL rows above")
print("=" * 58)
print(f"  Model selected : {MODEL}")
print(f"  Dataset        : {DATASET}")
print(f"  Repo           : {REPO_DIR} @ {BRANCH}")
for name, good, detail in checks:
    print(f"    {'OK  ' if good else 'FAIL'} {name}")
print("=" * 58)
print("Setup only — no training or benchmarking was run.")
print("Future benchmark phases: change MODEL in the Config cell, Run All, then")
print("add/run the benchmark cell for that phase. Nothing else needs editing.")
